# Mid-Block Expansion 细粒度观察

逐样本查看 **mask → token** 随 step 变化的完整过程。

**观察内容：**
- 每步哪些 position 从 [MASK] 变成了 token（heatmap）
- expansion 触发时的 block 边界变化
- 三组 config 在同一样本上的 decode 行为对比

**样本：** GSM8K x 5 + MBPP x 5

## 1. 环境设置

In [ ]:
import os
import torch
import gc

# Set GPU (modify as needed)
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5,6,7'

# Environment settings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

# Change to llada directory
os.chdir('llada')

# Clear GPU cache
torch.cuda.empty_cache()
gc.collect()

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. 加载模型

In [ ]:
from transformers import AutoTokenizer
from model.modeling_llada import LLaDAModelLM

device = 'cuda'
model_name = 'GSAI-ML/LLaDA-8B-Instruct'

print(f"Loading model: {model_name}")
model = LLaDAModelLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("Model loaded!")

## 3. 准备样本（GSM8K x 5 + MBPP x 5）

In [ ]:
from datasets import load_dataset

N_SAMPLES = 5

# --- GSM8K ---
ds_gsm = load_dataset("openai/gsm8k", "main", split="test")
gsm_samples = []
for i in range(N_SAMPLES):
    q = ds_gsm[i]['question']
    m = [{"role": "user", "content": q}]
    prompt_text = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
    input_ids = torch.tensor(tokenizer(prompt_text)['input_ids']).to(device).unsqueeze(0)
    gsm_samples.append({'idx': i, 'task': 'gsm8k', 'question': q[:80], 'input_ids': input_ids})
    print(f"GSM8K #{i}: {q[:60]}...")

# --- MBPP ---
ds_mbpp = load_dataset("google-research-datasets/mbpp", "full", split="test")
mbpp_samples = []
for i in range(N_SAMPLES):
    q = ds_mbpp[i]['text']
    m = [{"role": "user", "content": q}]
    prompt_text = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
    input_ids = torch.tensor(tokenizer(prompt_text)['input_ids']).to(device).unsqueeze(0)
    mbpp_samples.append({'idx': i, 'task': 'mbpp', 'question': q[:80], 'input_ids': input_ids})
    print(f"MBPP  #{i}: {q[:60]}...")

all_samples = gsm_samples + mbpp_samples
print(f"\nTotal samples: {len(all_samples)}")

## 4. 运行三组配置并收集 mask snapshot

In [ ]:
from generate import generate_with_dual_cache_expand
from tqdm.auto import tqdm
import numpy as np

GEN_LENGTH   = 256
STEPS        = 256
BLOCK_LENGTH = 32
THRESHOLD    = 0.9
MASK_ID      = 126336

configs = {
    'baseline':         {'mid_trigger_ratio': 0.0, 'rewarm_on_expand': True},
    'expand_no_rewarm': {'mid_trigger_ratio': 0.5, 'rewarm_on_expand': False},
    'expand_rewarm':    {'mid_trigger_ratio': 0.5, 'rewarm_on_expand': True},
}

# results[config_name][sample_idx] = {'step_records': [...], 'gen_text': str, 'nfe': int}
results = {c: [] for c in configs}

for cname, cfg in configs.items():
    print(f"\n{'='*60}")
    print(f"Config: {cname}")
    print(f"{'='*60}")
    for sample in tqdm(all_samples, desc=cname):
        input_ids = sample['input_ids']
        prompt_len = input_ids.shape[1]

        with torch.inference_mode():
            x, nfe, step_records = generate_with_dual_cache_expand(
                model, input_ids,
                steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
                temperature=0., threshold=THRESHOLD,
                mid_trigger_ratio=cfg['mid_trigger_ratio'],
                rewarm_on_expand=cfg['rewarm_on_expand'],
                record_steps=True,
            )

        gen_token_ids = x[0, prompt_len:].cpu().tolist()  # 保存完整 token ids
        gen_text = tokenizer.decode(gen_token_ids, skip_special_tokens=True)
        results[cname].append({
            'task': sample['task'],
            'idx': sample['idx'],
            'question': sample['question'],
            'gen_text': gen_text,
            'gen_token_ids': gen_token_ids,
            'nfe': nfe,
            'step_records': step_records,
        })

print("\nAll done.")

## 5. Mask 演化热力图

每张图：X 轴 = 生成区域的 position（0~255），Y 轴 = step 序号。
- **白色** = 已解码 token
- **深色** = 仍然是 [MASK]
- **红色竖线** = block 边界（每 32 个 token）
- **黄色标记** = expansion 触发的 step

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

config_labels = {
    'baseline': 'Baseline',
    'expand_no_rewarm': 'Expand (no rewarm)',
    'expand_rewarm': 'Expand (rewarm)',
}

def plot_mask_heatmap(step_records, title, ax):
    """绘制单个样本的 mask 演化热力图。"""
    snapshots = []
    step_types = []
    for r in step_records:
        snap = r.get('mask_snapshot')
        if snap is not None:
            snapshots.append(snap)
            step_types.append(r['type'])

    if not snapshots:
        ax.text(0.5, 0.5, 'No mask snapshots', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return

    mat = np.array(snapshots, dtype=np.float32)  # (n_steps, gen_length), 1=MASK 0=decoded
    n_steps, gen_len = mat.shape

    # 用自定义 colormap: 0=白(decoded), 1=深蓝(MASK)
    cmap = ListedColormap(['#FFFFFF', '#1565C0'])
    ax.imshow(mat, aspect='auto', cmap=cmap, interpolation='nearest', vmin=0, vmax=1)

    # Block 边界竖线
    for b in range(BLOCK_LENGTH, gen_len, BLOCK_LENGTH):
        ax.axvline(x=b - 0.5, color='red', linewidth=0.5, alpha=0.5)

    # 标记 expand step（黄色三角）
    for step_i, stype in enumerate(step_types):
        if stype == 'expand':
            ax.plot(-1, step_i, marker='>', color='#FF9800', markersize=6, clip_on=False)

    ax.set_xlabel('Position')
    ax.set_ylabel('Step')
    ax.set_title(title, fontsize=10)


# --- 为每个样本画三组对比 ---
os.makedirs('../eval_results', exist_ok=True)

for sample_i in range(len(all_samples)):
    sample_info = all_samples[sample_i]
    task_name = sample_info['task'].upper()
    q_short = sample_info['question'][:50]

    fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)

    for ax_i, cname in enumerate(configs.keys()):
        res = results[cname][sample_i]
        title = f"{config_labels[cname]}\nNFE={res['nfe']}"
        plot_mask_heatmap(res['step_records'], title, axes[ax_i])

    fig.suptitle(f"[{task_name} #{sample_info['idx']}] {q_short}...", fontsize=12, fontweight='bold')

    # Legend
    legend_elements = [
        mpatches.Patch(facecolor='#1565C0', label='[MASK]'),
        mpatches.Patch(facecolor='#FFFFFF', edgecolor='gray', label='Decoded'),
        plt.Line2D([0], [0], color='red', linewidth=1, label='Block boundary'),
        plt.Line2D([0], [0], marker='>', color='#FF9800', linestyle='None', markersize=8, label='Expand step'),
    ]
    fig.legend(handles=legend_elements, loc='upper right', fontsize=9, ncol=4)

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    fname = f"../eval_results/detail_{sample_info['task']}_{sample_info['idx']}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

## 6. 逐步 Diff 视图

每步新解码了哪些 position（上一步是 MASK → 本步变成 token 的位置）。
- **绿色** = 本步新解码
- **灰色** = 之前已解码
- **深蓝** = 仍是 MASK

In [ ]:
def plot_diff_heatmap(step_records, title, ax):
    """绘制逐步 diff 热力图：本步新解码的 position 用绿色高亮。"""
    snapshots = []
    for r in step_records:
        snap = r.get('mask_snapshot')
        if snap is not None:
            snapshots.append(snap)

    if not snapshots:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return

    mat = np.array(snapshots, dtype=np.int8)  # 1=MASK, 0=decoded
    n_steps, gen_len = mat.shape

    # 构造 diff 矩阵: 0=still_mask, 1=already_decoded, 2=newly_decoded
    diff_mat = np.zeros_like(mat, dtype=np.int8)
    for s in range(n_steps):
        if s == 0:
            diff_mat[s] = np.where(mat[s] == 1, 0, 2)  # 第0步: 非mask的都是"新解码"
        else:
            newly = (mat[s-1] == 1) & (mat[s] == 0)  # 上步是mask，本步不是
            already = (mat[s] == 0) & (~newly)         # 本步不是mask，也不是新解码
            diff_mat[s] = np.where(mat[s] == 1, 0, np.where(newly, 2, 1))

    # 0=MASK(深蓝), 1=already_decoded(浅灰), 2=newly_decoded(绿色)
    cmap = ListedColormap(['#1565C0', '#E0E0E0', '#4CAF50'])
    ax.imshow(diff_mat, aspect='auto', cmap=cmap, interpolation='nearest', vmin=0, vmax=2)

    for b in range(BLOCK_LENGTH, gen_len, BLOCK_LENGTH):
        ax.axvline(x=b - 0.5, color='red', linewidth=0.5, alpha=0.5)

    ax.set_xlabel('Position')
    ax.set_ylabel('Step')
    ax.set_title(title, fontsize=10)


for sample_i in range(len(all_samples)):
    sample_info = all_samples[sample_i]
    task_name = sample_info['task'].upper()
    q_short = sample_info['question'][:50]

    fig, axes = plt.subplots(1, 3, figsize=(22, 5), sharey=True)

    for ax_i, cname in enumerate(configs.keys()):
        res = results[cname][sample_i]
        title = f"{config_labels[cname]}\nNFE={res['nfe']}"
        plot_diff_heatmap(res['step_records'], title, axes[ax_i])

    fig.suptitle(f"[{task_name} #{sample_info['idx']}] Step Diff: {q_short}...", fontsize=12, fontweight='bold')

    legend_elements = [
        mpatches.Patch(facecolor='#1565C0', label='[MASK]'),
        mpatches.Patch(facecolor='#E0E0E0', label='Already decoded'),
        mpatches.Patch(facecolor='#4CAF50', label='Newly decoded this step'),
        plt.Line2D([0], [0], color='red', linewidth=1, label='Block boundary'),
    ]
    fig.legend(handles=legend_elements, loc='upper right', fontsize=9, ncol=4)

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    fname = f"../eval_results/diff_{sample_info['task']}_{sample_info['idx']}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

## 7. 每步解码数 + 累计进度（逐样本对比）

In [ ]:
colors_cfg = {'baseline': '#2196F3', 'expand_no_rewarm': '#FF9800', 'expand_rewarm': '#4CAF50'}

for sample_i in range(len(all_samples)):
    sample_info = all_samples[sample_i]
    task_name = sample_info['task'].upper()
    q_short = sample_info['question'][:50]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))

    for cname in configs.keys():
        res = results[cname][sample_i]
        sr = res['step_records']
        steps_x = [r['global_step'] for r in sr]
        transferred = [r['transferred'] for r in sr]
        cumulative = np.cumsum(transferred) / GEN_LENGTH

        color = colors_cfg[cname]
        label = f"{config_labels[cname]} (NFE={res['nfe']})"

        # 左图: 每步解码数
        ax1.bar([s + list(configs.keys()).index(cname) * 0.25 for s in steps_x],
                transferred, width=0.25, color=color, alpha=0.7, label=label)

        # 右图: 累计进度
        ax2.plot(steps_x, cumulative, color=color, linewidth=2, label=label, marker='.', markersize=3)

        # 标记 expand steps
        for r in sr:
            if r['type'] == 'expand':
                ax2.axvline(x=r['global_step'], color=color, linestyle='--', alpha=0.4)

    ax1.set_xlabel('Global Step')
    ax1.set_ylabel('Tokens Transferred')
    ax1.set_title('Per-Step Decode Count')
    ax1.legend(fontsize=8)

    ax2.set_xlabel('Global Step')
    ax2.set_ylabel('Cumulative Decoded Ratio')
    ax2.set_title('Decode Progress')
    ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.4)
    ax2.set_ylim(0, 1.1)
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    fig.suptitle(f"[{task_name} #{sample_info['idx']}] {q_short}...", fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()

## 8. 逐步解码文本流（线性输出）

选一个样本 + 一个 config，逐步展示 [MASK] → token 的完整演化。已解码的 token 用最终结果还原（token 一旦解码不会变回 MASK）。

In [ ]:
from IPython.display import display, HTML

MASK_TOKEN = '[mask]'

def render_step_text(mask_snapshot, final_token_ids, tokenizer, max_display_len=120):
    """根据 mask 位图 + 最终 token ids 还原某一步的文本。"""
    pieces = []
    for pos, is_mask in enumerate(mask_snapshot):
        if pos >= len(final_token_ids):
            break
        if is_mask:
            pieces.append(MASK_TOKEN)
        else:
            tok = tokenizer.decode([final_token_ids[pos]])
            pieces.append(tok)
    text = ''.join(pieces)
    # 截断显示
    if len(text) > max_display_len:
        text = text[:max_display_len] + '...'
    return text


def show_decode_flow(sample_i, config_name, max_steps_display=40):
    """打印单个样本在某个 config 下的逐步解码流。"""
    res = results[config_name][sample_i]
    sample_info = all_samples[sample_i]
    sr = res['step_records']

    # 直接用保存的 final token ids（精确）
    final_token_ids = res['gen_token_ids']

    print(f"{'='*100}")
    print(f"[{sample_info['task'].upper()} #{sample_info['idx']}] Config: {config_labels[config_name]}")
    print(f"Question: {sample_info['question']}")
    print(f"NFE: {res['nfe']}, Steps: {len(sr)}")
    print(f"{'='*100}")

    for step_i, r in enumerate(sr):
        if step_i >= max_steps_display:
            print(f"  ... ({len(sr) - max_steps_display} more steps omitted)")
            break

        snap = r.get('mask_snapshot')
        if snap is None:
            continue

        n_mask = sum(snap)
        n_decoded = len(snap) - n_mask
        step_type = r['type'].upper()
        transferred = r['transferred']

        # 用 mask_snapshot 还原文本
        text = render_step_text(snap, final_token_ids, tokenizer)

        # 颜色标记 step type
        type_tag = {'WARM': '🔥', 'REFINE': '🔧', 'EXPAND': '⚡'}.get(step_type, '  ')

        print(f"  Step {r['global_step']:>3} {type_tag} [+{transferred:>2} tok, {n_mask:>3} masks left]  {text}")

    print()


# === 展示所有样本在 baseline 下的解码流 ===
print("=" * 100)
print("BASELINE 解码流程")
print("=" * 100)
for i in range(len(all_samples)):
    show_decode_flow(i, 'baseline', max_steps_display=30)

## 9. 三组 Config 同一样本的解码流对比

选几个代表性样本，把三组配置的逐步解码放在一起看。

In [ ]:
# 选前 3 个样本（GSM8K #0, #1, MBPP #0），三组 config 逐步对比
compare_samples = [0, 1, 5]  # index into all_samples

for sample_i in compare_samples:
    sample_info = all_samples[sample_i]
    print(f"\n{'#'*100}")
    print(f"  [{sample_info['task'].upper()} #{sample_info['idx']}] {sample_info['question']}")
    print(f"{'#'*100}")

    for cname in configs.keys():
        show_decode_flow(sample_i, cname, max_steps_display=25)

## 10. 生成文本汇总对比

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 120)

rows = []
for sample_i in range(len(all_samples)):
    sample_info = all_samples[sample_i]
    row = {
        'task': sample_info['task'],
        'idx': sample_info['idx'],
        'question': sample_info['question'][:60],
    }
    for cname in configs.keys():
        res = results[cname][sample_i]
        n_expand = sum(1 for r in res['step_records'] if r['type'] == 'expand')
        row[f'{cname}_nfe'] = res['nfe']
        row[f'{cname}_expands'] = n_expand
        row[f'{cname}_text'] = res['gen_text'][:100]
    rows.append(row)

df_detail = pd.DataFrame(rows)

# NFE + expand 汇总
summary_cols = ['task', 'idx', 'question']
for c in configs.keys():
    summary_cols += [f'{c}_nfe', f'{c}_expands']
display(df_detail[summary_cols])

# 逐样本生成文本
for _, row in df_detail.iterrows():
    print(f"\n{'='*80}")
    print(f"[{row['task'].upper()} #{row['idx']}] {row['question']}")
    print(f"{'='*80}")
    for cname in configs.keys():
        print(f"  {config_labels[cname]} (NFE={row[f'{cname}_nfe']}, expands={row[f'{cname}_expands']}):")
        print(f"    {row[f'{cname}_text']}")
    print()